# 02 - Feature Set Data Diagnostics

This notebook compares biological-age recovery across observation designs on the combined baseline dataset.

The network/training setup is held fixed while the feature set changes. This separates a **data-information question** from the network diagnostics in notebook `01`:

- Does adding indicators improve synthetic recovery?
- Does adding indicators improve real baseline holdout recovery?
- Does predictive uncertainty shrink as the feature set becomes richer?

Here `r` is Pearson correlation between actual biological age and the predicted mean biological age. It measures association, not calibration.


## Setup

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path
from datetime import datetime
import copy
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import keras
import bayesflow as bf

SRC_CANDIDATES = [Path("../src"), Path("biological_age_sbi/experiment/src")]
for src in SRC_CANDIDATES:
    if (src / "bioage_sbi").exists():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError("Could not find biological_age_sbi/experiment/src")

from bioage_sbi import (
    FEATURE_SETS,
    LATENT_KEYS,
    find_baseline_data_path,
    fit_empirical_model,
    load_baseline_data,
    prepare_model_frame,
    save_empirical_model,
    split_train_holdout,
)
from bioage_sbi.simulator import make_component_functions, sample_component_model

sns.set_theme(style="whitegrid")
np.set_printoptions(suppress=True)

SEED = 1234
SPLIT_SEED = 2026
TRAINING_SEED_BASE = SEED + 100
VALIDATION_SEED_BASE = SEED + 200
NETWORK_SEED_BASE = SEED + 300
SYNTHETIC_EVAL_SEED_BASE = SEED + 400

DATA_DIAGNOSTIC_FEATURE_SETS = [
    "set_a_easy_non_lab",
    "set_f_common_non_lab_diagnosis",
    "set_e_kdm8_common_lab",
]

SIMULATOR_VARIANT = "copula_empirical_residuals_sbp_bmi_0_5_sbp_agebin_mean_0_5"
SIMULATOR_DESCRIPTION = "Conditional Gaussian copula enabled; empirical continuous residual bootstrap enabled; SBP-BMI effect scaled by 0.5; train-only SBP age-bin mean correction scaled by 0.5."
COPULA_ENABLED = True
CONTINUOUS_RESIDUAL_NOISE_SCALE = 1.0
OBSERVATION_NOISE_SCALE = 1.0
LATENT_STD_SCALE = 1.0
LATENT_AGE_EFFECT_SCALE = 1.0
LATENT_LOADING_SCALE = 1.0
CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP = True
SBP_BMI_EFFECT_SCALE = 0.5
SBP_AGE_BIN_MEAN_CORRECTION = True
SBP_AGE_BIN_MEAN_CORRECTION_SCALE = 0.5
AGE_BIN_CALIBRATION_SYNTHETIC_N = None  # None means match train size.
AGE_BINS = np.arange(40, 101, 10)

DIAGNOSTIC_BATCH_COUNTS = [2, 8, 32]
DIAGNOSTIC_EPOCHS_BY_BATCH_COUNT = {2: 300, 8: 200, 32: 120}
BATCH_SIZE = 128
NUM_VALIDATION_SIMS = 1000
SYNTHETIC_EVAL_N = 500

MLP_WIDTHS = (512, 512, 256)
MLP_NORM = "layer"
MLP_DROPOUT = 0.05
MLP_RESIDUAL = True
q_levels = np.linspace(0.1, 0.9, 5)


def reset_random_seeds(seed=SEED):
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


## Result Folders

In [ ]:
DATA_PATH = find_baseline_data_path()
EXPERIMENT_DIR = DATA_PATH.parents[2]
PROCESSED_DIR = DATA_PATH.parent
DATA_DIAGNOSTICS_DIR = EXPERIMENT_DIR / "results" / "data_diagnostics"
NUMBER_OF_FEATURES_DIR = DATA_DIAGNOSTICS_DIR / "number_of_features"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NUMBER_OF_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_OUTPUT_DIR = NUMBER_OF_FEATURES_DIR / f"{RUN_TIMESTAMP}_{SIMULATOR_VARIANT}"
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

FEATURE_SET_SUMMARY_PATH = RUN_OUTPUT_DIR / "feature_set_summary.csv"
FEATURE_SET_RESULTS_PATH = RUN_OUTPUT_DIR / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}.csv"
FEATURE_SET_LOSS_HISTORY_PATH = RUN_OUTPUT_DIR / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}_loss_history.csv"
FEATURE_SET_METRICS_FIGURE_PATH = RUN_OUTPUT_DIR / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}_metrics.png"
FEATURE_SET_LOSS_FIGURE_PATH = RUN_OUTPUT_DIR / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}_loss_history.png"
FEATURE_SET_CERTAINTY_FIGURE_PATH = RUN_OUTPUT_DIR / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}_certainty.png"

RUN_OUTPUT_DIR


## Feature Set Ladder

In [ ]:
feature_set_table = pd.DataFrame(
    [
        {
            "feature_set": name,
            "description": spec.description,
            "continuous_count": len(spec.continuous_columns),
            "binary_count": len(spec.binary_columns),
            "feature_count": len(spec.columns),
            "continuous_columns": ", ".join(spec.continuous_columns),
            "binary_columns": ", ".join(spec.binary_columns),
        }
        for name, spec in FEATURE_SETS.items()
        if name in DATA_DIAGNOSTIC_FEATURE_SETS
    ]
)
feature_set_table


## Training Helpers

In [ ]:
def apply_simulator_variant(model):
    active_model = copy.deepcopy(model)

    for spec in active_model["continuous_models"].values():
        spec["residual_std"] = float(spec["residual_std"] * CONTINUOUS_RESIDUAL_NOISE_SCALE)

    continuous_noise = active_model["observation_model"]["continuous_noise_std"]
    for key, value in continuous_noise.items():
        continuous_noise[key] = float(value * OBSERVATION_NOISE_SCALE)

    dependence_model = active_model.get("dependence_model")
    if dependence_model is not None:
        dependence_model["enabled"] = bool(COPULA_ENABLED)

    latent_config = active_model.get("latent_factors", {})
    for spec in latent_config.get("factors", {}).values():
        spec["std"] = float(spec["std"] * LATENT_STD_SCALE)
        if "age_z" in spec.get("mean_terms", {}):
            spec["mean_terms"]["age_z"] = float(spec["mean_terms"]["age_z"] * LATENT_AGE_EFFECT_SCALE)

    for loadings_by_variable in [
        latent_config.get("continuous_loadings", {}),
        latent_config.get("binary_logit_loadings", {}),
    ]:
        for loadings in loadings_by_variable.values():
            for key, value in loadings.items():
                loadings[key] = float(value * LATENT_LOADING_SCALE)

    continuous_calibration = active_model.get("calibration", {}).get("continuous")
    if continuous_calibration is not None:
        continuous_calibration["enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration["empirical_residual_bootstrap_enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration.setdefault("effect_scales", {})["sbp_bmi"] = float(SBP_BMI_EFFECT_SCALE)

    active_model["simulator_variant"] = {
        "name": SIMULATOR_VARIANT,
        "description": SIMULATOR_DESCRIPTION,
        "copula_enabled": bool(active_model.get("dependence_model", {}).get("enabled", False)),
        "continuous_residual_noise_scale": CONTINUOUS_RESIDUAL_NOISE_SCALE,
        "observation_noise_scale": OBSERVATION_NOISE_SCALE,
        "latent_std_scale": LATENT_STD_SCALE,
        "latent_age_effect_scale": LATENT_AGE_EFFECT_SCALE,
        "latent_loading_scale": LATENT_LOADING_SCALE,
        "continuous_empirical_residual_bootstrap": bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP),
        "sbp_bmi_effect_scale": SBP_BMI_EFFECT_SCALE,
        "sbp_age_bin_mean_correction": bool(SBP_AGE_BIN_MEAN_CORRECTION),
        "sbp_age_bin_mean_correction_scale": SBP_AGE_BIN_MEAN_CORRECTION_SCALE,
    }
    return active_model


def synthetic_observed_frame(model, num_samples, seed):
    simulated = sample_component_model(model, num_samples=num_samples, seed=seed)
    observed_columns = [model["observed_key_by_column"][col] for col in model["columns"]]
    rename = {model["observed_key_by_column"][col]: col for col in model["columns"]}
    return simulated[["biological_age", *observed_columns]].rename(columns=rename)


def calibrate_sbp_age_bin_mean(model, train_frame, seed):
    if not SBP_AGE_BIN_MEAN_CORRECTION or "sbp" not in model["columns"]:
        return model

    continuous_calibration = model.get("calibration", {}).get("continuous")
    if continuous_calibration is None:
        return model

    n_synthetic = len(train_frame) if AGE_BIN_CALIBRATION_SYNTHETIC_N is None else AGE_BIN_CALIBRATION_SYNTHETIC_N
    synthetic_train = synthetic_observed_frame(model, n_synthetic, seed=seed)
    real_train = train_frame[["biological_age", "sbp"]].copy()

    synthetic_bins = pd.cut(synthetic_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    real_bins = pd.cut(real_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    adjustments = []
    for interval in synthetic_bins.cat.categories:
        synthetic_mean = synthetic_train.loc[synthetic_bins == interval, "sbp"].mean()
        real_mean = real_train.loc[real_bins == interval, "sbp"].mean()
        if np.isfinite(synthetic_mean) and np.isfinite(real_mean):
            adjustment = (real_mean - synthetic_mean) * SBP_AGE_BIN_MEAN_CORRECTION_SCALE
        else:
            adjustment = 0.0
        adjustments.append(float(adjustment))

    age_adjustment = continuous_calibration.setdefault("age_bin_mean_adjustment", {})
    age_adjustment["enabled"] = True
    age_adjustment.setdefault("columns", {})["sbp"] = adjustments
    age_adjustment["source"] = "train_split_synthetic_minus_real_age_bin_mean"
    age_adjustment["scale"] = float(SBP_AGE_BIN_MEAN_CORRECTION_SCALE)
    age_adjustment["synthetic_n"] = int(n_synthetic)
    return model


def build_simulator(model, seed):
    prior, bioindicator_model, observation_model = make_component_functions(model, seed=seed)
    simulator = bf.make_simulator([prior, bioindicator_model, observation_model])
    return prior, bioindicator_model, observation_model, simulator


def model_true_keys(model):
    return [model.get("true_key_by_column", {}).get(col, f"true_{col}") for col in model["columns"]]


def make_adapter(model):
    condition_mean = np.asarray(model["adapter"]["mean"], dtype="float32")
    condition_std = np.asarray(model["adapter"]["std"], dtype="float32")
    return (
        bf.adapters.Adapter()
        .convert_dtype("float64", "float32")
        .drop([*LATENT_KEYS, *model_true_keys(model)])
        .concatenate(["biological_age"], into="inference_variables")
        .concatenate(model["adapter"]["condition_keys"], into="inference_conditions")
        .standardize("inference_conditions", mean=condition_mean, std=condition_std)
    )


def frame_to_conditions(model, frame):
    conditions = {
        model["observed_key_by_column"][col]: frame[col].to_numpy(dtype="float64").reshape(-1, 1)
        for col in model["columns"]
    }
    conditions["biological_age"] = frame["biological_age"].to_numpy(dtype="float64").reshape(-1, 1)
    return conditions


def extract_bioage_estimates(estimates):
    bioage_estimates = estimates.get("biological_age", estimates.get("inference_variables"))
    mean = np.asarray(bioage_estimates["mean"]).squeeze()
    quantiles = np.asarray(bioage_estimates["quantiles"]).squeeze()
    return mean, quantiles


def prediction_metrics(actual, mean, quantiles=None):
    actual = np.asarray(actual).squeeze()
    mean = np.asarray(mean).squeeze()
    error = mean - actual
    metrics = {
        "correlation_r": np.corrcoef(actual, mean)[0, 1],
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error**2)),
        "mean_error": np.mean(error),
    }
    if quantiles is not None:
        metrics["mean_q10_q90_width"] = np.mean(quantiles[:, -1] - quantiles[:, 0])
        metrics["q10_q90_coverage"] = np.mean((actual >= quantiles[:, 0]) & (actual <= quantiles[:, -1]))
    return pd.Series(metrics)


def history_to_loss_frame(history, run_metadata):
    history_dict = getattr(history, "history", history)
    if isinstance(history_dict, pd.DataFrame):
        history_dict = history_dict.to_dict(orient="list")
    if not isinstance(history_dict, dict):
        return pd.DataFrame()

    max_len = max((len(values) for values in history_dict.values() if hasattr(values, "__len__")), default=0)
    rows = []
    for epoch_index in range(max_len):
        row = {**run_metadata, "epoch": epoch_index + 1}
        for key, values in history_dict.items():
            if hasattr(values, "__len__") and epoch_index < len(values):
                try:
                    row[key] = float(values[epoch_index])
                except (TypeError, ValueError):
                    row[key] = values[epoch_index]
        rows.append(row)
    return pd.DataFrame(rows)


def final_training_loss_from_history(history):
    loss_frame = history_to_loss_frame(history, {})
    candidate_keys = ["loss", "train_loss", "training_loss"]
    candidate_keys.extend(key for key in loss_frame.columns if key.endswith("loss") and not key.startswith("val"))
    loss_key = next((key for key in candidate_keys if key in loss_frame.columns), None)
    if loss_key is None or loss_frame.empty:
        return {"loss_key": "unavailable", "first_train_loss": np.nan, "final_train_loss": np.nan, "min_train_loss": np.nan}
    return {
        "loss_key": loss_key,
        "first_train_loss": float(loss_frame[loss_key].iloc[0]),
        "final_train_loss": float(loss_frame[loss_key].iloc[-1]),
        "min_train_loss": float(loss_frame[loss_key].min()),
    }


## Fit And Save Empirical Models

Each feature set gets its own prepared frame, deterministic train/holdout split, empirical simulator JSON, and train/holdout CSV files.


In [ ]:
raw_df = load_baseline_data(DATA_PATH)

feature_set_artifacts = {}
feature_set_summary_rows = []

for feature_index, feature_set_name in enumerate(DATA_DIAGNOSTIC_FEATURE_SETS):
    model_frame = prepare_model_frame(raw_df, feature_set_name=feature_set_name)
    train_frame, holdout_frame = split_train_holdout(model_frame, holdout_fraction=0.20, seed=SPLIT_SEED)
    empirical_model = fit_empirical_model(
        train_frame,
        model_name=feature_set_name,
        feature_set_name=feature_set_name,
    )
    active_model = calibrate_sbp_age_bin_mean(
        apply_simulator_variant(empirical_model),
        train_frame,
        seed=TRAINING_SEED_BASE + 10_000 + feature_index,
    )

    model_path = PROCESSED_DIR / f"empirical_bioage_model_{feature_set_name}.json"
    train_path = PROCESSED_DIR / f"baseline_train_{feature_set_name}.csv"
    holdout_path = PROCESSED_DIR / f"baseline_holdout_{feature_set_name}.csv"

    save_empirical_model(empirical_model, model_path)
    train_frame.to_csv(train_path, index=False)
    holdout_frame.to_csv(holdout_path, index=False)

    feature_set_artifacts[feature_set_name] = {
        "model": empirical_model,
        "active_model": active_model,
        "train_frame": train_frame,
        "holdout_frame": holdout_frame,
        "model_path": model_path,
        "train_path": train_path,
        "holdout_path": holdout_path,
    }
    feature_set_summary_rows.append(
        {
            "feature_set": feature_set_name,
            "description": empirical_model["feature_set_description"],
            "n_total_complete": len(model_frame),
            "n_train": len(train_frame),
            "n_holdout": len(holdout_frame),
            "continuous_count": len(empirical_model["continuous_columns"]),
            "binary_count": len(empirical_model["binary_columns"]),
            "feature_count": len(empirical_model["columns"]),
            "model_path": str(model_path),
            "train_path": str(train_path),
            "holdout_path": str(holdout_path),
            "simulator_variant": SIMULATOR_VARIANT,
            "run_timestamp": RUN_TIMESTAMP,
            "run_output_dir": str(RUN_OUTPUT_DIR),
        }
    )

feature_set_summary = pd.DataFrame(feature_set_summary_rows)
feature_set_summary.to_csv(FEATURE_SET_SUMMARY_PATH, index=False)
feature_set_summary


## Prior Predictive Sanity Check

This checks broad simulator output shapes and dependence summaries before training. It is not the final evaluation.


In [ ]:
prior_predictive_rows = []
for feature_set_name, artifact in feature_set_artifacts.items():
    model = artifact["active_model"]
    train_frame = artifact["train_frame"]
    simulated_frame = sample_component_model(model, num_samples=min(2000, len(train_frame)), seed=SEED)
    observed_columns = [model["observed_key_by_column"][col] for col in model["columns"]]
    simulated_observed = simulated_frame[["biological_age", *observed_columns]].rename(
        columns={model["observed_key_by_column"][col]: col for col in model["columns"]}
    )
    common_columns = ["biological_age", *model["columns"]]
    corr_gap = (train_frame[common_columns].corr() - simulated_observed[common_columns].corr()).abs().to_numpy()
    prior_predictive_rows.append(
        {
            "feature_set": feature_set_name,
            "feature_count": len(model["columns"]),
            "mean_abs_corr_gap": np.nanmean(corr_gap),
            "max_abs_corr_gap": np.nanmax(corr_gap),
        }
    )

prior_predictive_summary = pd.DataFrame(prior_predictive_rows)
prior_predictive_summary


## Run Number-Of-Features Batch-Count Diagnostic

This trains one fresh network for every feature set and batch count in `[2, 8, 32]`. These are the network sizes where the loss-path diagnostic remained interpretable. The result grid lets us compare whether adding features changes learnability at the same simulation budget.


In [ ]:
data_diagnostic_rows = []
loss_history_frames = []

for feature_index, feature_set_name in enumerate(DATA_DIAGNOSTIC_FEATURE_SETS):
    artifact = feature_set_artifacts[feature_set_name]
    active_model = artifact["active_model"]
    adapter = make_adapter(active_model)
    real_conditions = frame_to_conditions(active_model, artifact["holdout_frame"])
    real_actual = artifact["holdout_frame"]["biological_age"].to_numpy()

    for batch_index, batch_count in enumerate(DIAGNOSTIC_BATCH_COUNTS):
        run_index = feature_index * len(DIAGNOSTIC_BATCH_COUNTS) + batch_index
        epochs = DIAGNOSTIC_EPOCHS_BY_BATCH_COUNT[batch_count]
        num_training_sims = batch_count * BATCH_SIZE

        training_seed = TRAINING_SEED_BASE + run_index
        validation_seed = VALIDATION_SEED_BASE + run_index
        network_seed = NETWORK_SEED_BASE + run_index
        synthetic_eval_seed = SYNTHETIC_EVAL_SEED_BASE + run_index

        _, _, _, training_simulator = build_simulator(active_model, training_seed)
        _, _, _, validation_simulator = build_simulator(active_model, validation_seed)
        _, _, _, synthetic_eval_simulator = build_simulator(active_model, synthetic_eval_seed)

        training_data = training_simulator.sample(num_training_sims)
        validation_data = validation_simulator.sample(NUM_VALIDATION_SIMS)
        synthetic_eval_data = synthetic_eval_simulator.sample(SYNTHETIC_EVAL_N)

        reset_random_seeds(network_seed)
        keras.backend.clear_session()
        network = bf.networks.ScoringRuleNetwork(
            scoring_rules=dict(
                mean=bf.scoring_rules.MeanScore(),
                quantiles=bf.scoring_rules.QuantileScore(q_levels),
            ),
            subnet="mlp",
            subnet_kwargs=dict(
                widths=MLP_WIDTHS,
                norm=MLP_NORM,
                dropout=MLP_DROPOUT,
                residual=MLP_RESIDUAL,
            ),
        )
        workflow = bf.BasicWorkflow(
            simulator=training_simulator,
            adapter=adapter,
            inference_network=network,
        )
        history = workflow.fit_offline(
            training_data,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            validation_data=validation_data,
            verbose=0,
        )

        synthetic_estimates = workflow.approximator.estimate(conditions=synthetic_eval_data)
        synthetic_mean, synthetic_quantiles = extract_bioage_estimates(synthetic_estimates)
        synthetic_actual = synthetic_eval_data["biological_age"].squeeze()
        synthetic_metrics = prediction_metrics(synthetic_actual, synthetic_mean, synthetic_quantiles)

        real_estimates = workflow.approximator.estimate(conditions=real_conditions)
        real_mean, real_quantiles = extract_bioage_estimates(real_estimates)
        real_metrics = prediction_metrics(real_actual, real_mean, real_quantiles)

        run_metadata = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "approach_name": "number_of_features_batch_count_grid",
            "run_timestamp": RUN_TIMESTAMP,
            "run_output_dir": str(RUN_OUTPUT_DIR),
            "feature_set": feature_set_name,
            "feature_set_description": active_model["feature_set_description"],
            "simulator_variant": SIMULATOR_VARIANT,
            "model_version": active_model.get("version"),
            "model_family": active_model.get("model_family"),
            "feature_count": len(active_model["columns"]),
            "continuous_count": len(active_model["continuous_columns"]),
            "binary_count": len(active_model["binary_columns"]),
            "batch_count": batch_count,
            "num_training_sims": num_training_sims,
            "num_validation_sims": NUM_VALIDATION_SIMS,
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
            "mlp_widths": str(MLP_WIDTHS),
            "mlp_norm": MLP_NORM,
            "mlp_dropout": MLP_DROPOUT,
            "mlp_residual": MLP_RESIDUAL,
            "training_seed": training_seed,
            "validation_seed": validation_seed,
            "network_seed": network_seed,
            "synthetic_eval_seed": synthetic_eval_seed,
        }
        loss_metrics = final_training_loss_from_history(history)
        row = {
            **run_metadata,
            **loss_metrics,
            "synthetic_r": float(synthetic_metrics["correlation_r"]),
            "synthetic_mae": float(synthetic_metrics["mae"]),
            "synthetic_rmse": float(synthetic_metrics["rmse"]),
            "synthetic_mean_error": float(synthetic_metrics["mean_error"]),
            "synthetic_q10_q90_coverage": float(synthetic_metrics["q10_q90_coverage"]),
            "synthetic_mean_q10_q90_width": float(synthetic_metrics["mean_q10_q90_width"]),
            "real_r": float(real_metrics["correlation_r"]),
            "real_mae": float(real_metrics["mae"]),
            "real_rmse": float(real_metrics["rmse"]),
            "real_mean_error": float(real_metrics["mean_error"]),
            "real_q10_q90_coverage": float(real_metrics["q10_q90_coverage"]),
            "real_mean_q10_q90_width": float(real_metrics["mean_q10_q90_width"]),
        }
        data_diagnostic_rows.append(row)

        loss_history = history_to_loss_frame(history, run_metadata)
        if not loss_history.empty:
            loss_history_frames.append(loss_history)
            pd.concat(loss_history_frames, ignore_index=True).to_csv(FEATURE_SET_LOSS_HISTORY_PATH, index=False)

        data_diagnostic_results = pd.DataFrame(data_diagnostic_rows)
        data_diagnostic_results.to_csv(FEATURE_SET_RESULTS_PATH, index=False)
        display(data_diagnostic_results.tail(1))

data_diagnostic_results = pd.DataFrame(data_diagnostic_rows)
data_diagnostic_results.to_csv(FEATURE_SET_RESULTS_PATH, index=False)
if loss_history_frames:
    data_diagnostic_loss_history = pd.concat(loss_history_frames, ignore_index=True)
else:
    data_diagnostic_loss_history = pd.DataFrame()
data_diagnostic_loss_history.to_csv(FEATURE_SET_LOSS_HISTORY_PATH, index=False)

data_diagnostic_results.round(3)


## Plot Number-Of-Features Diagnostics

The first plot mirrors the network diagnostic: final/min training loss and Pearson `r` against number of training simulations. Lines are grouped by feature set, so each point is one trained network. The second plot shows per-epoch loss paths for every feature set and batch count.


In [ ]:
plot_results = data_diagnostic_results.copy()
feature_order = DATA_DIAGNOSTIC_FEATURE_SETS
plot_results["feature_label"] = pd.Categorical(plot_results["feature_set"], categories=feature_order, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(
    data=plot_results,
    x="num_training_sims",
    y="final_train_loss",
    hue="feature_set",
    marker="o",
    ax=axes[0],
)
sns.lineplot(
    data=plot_results,
    x="num_training_sims",
    y="min_train_loss",
    hue="feature_set",
    marker="o",
    linestyle="--",
    legend=False,
    ax=axes[0],
)
axes[0].set_xscale("log")
if (plot_results[["final_train_loss", "min_train_loss"]] > 0).all().all():
    axes[0].set_yscale("log")
axes[0].set_xlabel("Training simulations")
axes[0].set_ylabel("Training loss")
axes[0].set_title("Training loss versus data size")

sns.lineplot(
    data=plot_results,
    x="num_training_sims",
    y="synthetic_r",
    hue="feature_set",
    marker="o",
    ax=axes[1],
)
sns.lineplot(
    data=plot_results,
    x="num_training_sims",
    y="real_r",
    hue="feature_set",
    marker="X",
    linestyle="--",
    legend=False,
    ax=axes[1],
)
axes[1].set_xscale("log")
axes[1].set_ylim(-0.05, 1.0)
axes[1].set_xlabel("Training simulations")
axes[1].set_ylabel("Pearson r")
axes[1].set_title("Solid: synthetic r; dashed: real r")

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(FEATURE_SET_METRICS_FIGURE_PATH, dpi=160, bbox_inches="tight")

if not data_diagnostic_loss_history.empty:
    loss_key_for_plot = plot_results["loss_key"].dropna().iloc[0]
    grid = sns.relplot(
        data=data_diagnostic_loss_history,
        x="epoch",
        y=loss_key_for_plot,
        hue="batch_count",
        col="feature_set",
        col_wrap=2,
        kind="line",
        height=3.2,
        facet_kws={"sharex": False, "sharey": False},
    )
    for ax in grid.axes.flat:
        if loss_key_for_plot in data_diagnostic_loss_history and (data_diagnostic_loss_history[loss_key_for_plot] > 0).all():
            ax.set_yscale("log")
        sns.despine(ax=ax)
    grid.set_axis_labels("Epoch", loss_key_for_plot)
    grid.fig.suptitle("Training loss convergence by feature set and batch count", y=1.02)
    grid.fig.savefig(FEATURE_SET_LOSS_FIGURE_PATH, dpi=160, bbox_inches="tight")

certainty_long = plot_results.melt(
    id_vars=["feature_set", "feature_count", "batch_count", "num_training_sims"],
    value_vars=["synthetic_mean_q10_q90_width", "real_mean_q10_q90_width"],
    var_name="evaluation",
    value_name="mean_q10_q90_width",
)
certainty_long["evaluation"] = certainty_long["evaluation"].map(
    {
        "synthetic_mean_q10_q90_width": "synthetic",
        "real_mean_q10_q90_width": "real",
    }
)
fig_certainty, ax_certainty = plt.subplots(figsize=(9, 4))
sns.lineplot(
    data=certainty_long,
    x="feature_count",
    y="mean_q10_q90_width",
    hue="evaluation",
    style="batch_count",
    marker="o",
    ax=ax_certainty,
)
ax_certainty.set_xlabel("Number of observed features")
ax_certainty.set_ylabel("Mean q10-q90 width")
ax_certainty.set_title("Certainty proxy versus number of features")
sns.despine(ax=ax_certainty)
plt.tight_layout()
fig_certainty.savefig(FEATURE_SET_CERTAINTY_FIGURE_PATH, dpi=160, bbox_inches="tight")

print(f"Saved feature-set summary to: {FEATURE_SET_SUMMARY_PATH}")
print(f"Saved feature-set grid results to: {FEATURE_SET_RESULTS_PATH}")
print(f"Saved feature-set loss history to: {FEATURE_SET_LOSS_HISTORY_PATH}")
print(f"Saved metrics figure to: {FEATURE_SET_METRICS_FIGURE_PATH}")
print(f"Saved loss-path figure to: {FEATURE_SET_LOSS_FIGURE_PATH}")
print(f"Saved certainty figure to: {FEATURE_SET_CERTAINTY_FIGURE_PATH}")
data_diagnostic_results.round(3)


## Interpretation Guide

- Each row is one trained network: one feature set at one training simulation budget.
- If richer feature sets improve `synthetic_r` at the same batch count, the simulator contains more age-recoverable information.
- If richer feature sets improve `synthetic_r` but not `real_r`, transfer to the baseline holdout is the bottleneck.
- If loss decreases with more batches but `r` does not, the additional data may reduce memorization without adding useful age signal.
- If q10-q90 width shrinks as the number of features grows, the model becomes more certain under its scoring-rule outputs. This is only a certainty proxy, not a calibrated posterior interval.
